In [38]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy

import config as CFG
from models import CLIPModel, CLIPModel_ViT, CLIPModel_ViT_L, CLIPModel_CLIP, CLIPModel_resnet152, CLIPModel_resnet101
from dataset import CLIPDataset
import scanpy as sc
from torch.utils.data import DataLoader

import os
import numpy as np
import scanpy as sc

In [39]:
#print the current scanpy version
print(sc.__version__)

/tmp/ipykernel_2270147/3776245242.py:2: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(sc.__version__)


1.11.5


In [40]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
# 假设你的 CLIPDataset 和 CLIPModel 已经正确 import

def build_loaders_inference():
    print("Building loaders for all 4 slices...")
    # 🚨 注意：这里强制加上 is_train=False，防止测试集图片被随机翻转！
    
    dataset1 = CLIPDataset(image_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/images/GEX_C73_A1_Merged.tif",
                spatial_pos_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/tissue_pos_matrices/tissue_positions_list_1.csv",
                reduced_mtx_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/1/harmony_matrix.npy",
                barcode_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/1/barcodes.tsv",
                is_train=False)
    
    dataset2 = CLIPDataset(image_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/images/GEX_C73_B1_Merged.tif",
                spatial_pos_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/tissue_pos_matrices/tissue_positions_list_2.csv",
                reduced_mtx_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/2/harmony_matrix.npy",
                barcode_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/2/barcodes.tsv",
                is_train=False)
    
    dataset3 = CLIPDataset(image_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/images/GEX_C73_C1_Merged.tif",
                spatial_pos_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/tissue_pos_matrices/tissue_positions_list_3.csv",
                reduced_mtx_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/harmony_matrix.npy",
                barcode_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/barcodes.tsv",
                is_train=False)
    
    dataset4 = CLIPDataset(image_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/images/GEX_C73_D1_Merged.tif",
                spatial_pos_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/tissue_pos_matrices/tissue_positions_list_4.csv",
                reduced_mtx_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/4/harmony_matrix.npy",
                barcode_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/4/barcodes.tsv",
                is_train=False)
    
    # 把 1, 2, 3, 4 全部拼在一起
    dataset = torch.utils.data.ConcatDataset([dataset1, dataset2, dataset3, dataset4])
    
    # 返回单一的 DataLoader，这样 tqdm(test_loader) 就不会报错了
    test_loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, drop_last=False)

    print("Finished building loaders")
    return test_loader


def load_trained_model(model_path, model):
    """单独拿出一个函数来加载权重，避免重复加载"""
    state_dict = torch.load(model_path, map_location="cuda")
    new_state_dict = {}
    for key in state_dict.keys():
        new_key = key.replace('module.', '')  # 去除 DDP 留下的 module. 前缀
        new_key = new_key.replace('well', 'spot') 
        new_state_dict[new_key] = state_dict[key]

    model.load_state_dict(new_state_dict)
    model.cuda()
    model.eval()
    print("Finished loading model weights.")
    return model


def extract_embeddings(model, dataloader):
    """一次性将图像特征和基因特征全部提出来"""
    image_embeddings = []
    spot_embeddings = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting embeddings"):
            # 提取图像潜向量 (256维)
            img_feats = model.image_encoder(batch["image"].cuda())
            img_embeds = model.image_projection(img_feats)
            image_embeddings.append(img_embeds.cpu())
            
            # 提取基因表达潜向量 (256维)
            spot_embeds = model.spot_projection(batch["reduced_expression"].cuda())
            spot_embeddings.append(spot_embeds.cpu())
            
    return torch.cat(image_embeddings), torch.cat(spot_embeddings)


def find_matches(spot_embeddings, query_embeddings, top_k=50):
    """
    K-NN 检索：计算 Query 和 Reference 之间的余弦相似度
    注意：在 BLEEP 论文中，K 近邻的默认值通常是 50
    """
    # 🚨 修复：把 NumPy 数组转换为 PyTorch Tensor，并确保是 float32 类型
    query_tensor = torch.tensor(query_embeddings).float()
    spot_tensor = torch.tensor(spot_embeddings).float()
    
    # 然后再送给 F.normalize 算余弦相似度
    query_norm = F.normalize(query_tensor, p=2, dim=-1).cuda()
    spot_norm = F.normalize(spot_tensor, p=2, dim=-1).cuda()
    
    # 矩阵乘法计算相似度：(Query_N, 256) @ (256, Ref_N) -> (Query_N, Ref_N)
    dot_similarity = query_norm @ spot_norm.T
    print(f"Similarity matrix shape: {dot_similarity.shape}")
    
    # 找出每个 Query 图像最相似的 top_k 个 Reference 基因表达
    _, indices = torch.topk(dot_similarity, k=top_k, dim=1)
    
    return indices.cpu().numpy()

In [41]:
import os
import numpy as np
from utils import load_model_checkpoint

# 1. 基础设置
datasize = [2378, 2349, 2277, 2265]
model_path = "/root/disk2/runzhi/BLEEP/bleep_baseline_uni/best.pt"
save_path = "/root/disk2/runzhi/BLEEP/result/embeddings/"

USE_SPOT_ENCODER = False  # baseline 训练时应为 False

def load_trained_model(model_path, model):
    load_info = load_model_checkpoint(
        model=model,
        checkpoint_path=model_path,
        map_location="cuda",
        strict=False,
        rename_map={"well": "spot"},
    )

    ckpt_obj = load_info.get("checkpoint_obj", {})
    ckpt_args = ckpt_obj.get("args", {}) if isinstance(ckpt_obj, dict) else {}
    if isinstance(ckpt_args, dict) and ("use_spot_encoder" in ckpt_args):
        ckpt_use_spot_encoder = bool(ckpt_args["use_spot_encoder"])
        model_use_spot_encoder = bool(getattr(model, "use_spot_encoder", False))
        if ckpt_use_spot_encoder != model_use_spot_encoder:
            print(
                f"[Warning] checkpoint use_spot_encoder={ckpt_use_spot_encoder}, "
                f"but current model use_spot_encoder={model_use_spot_encoder}."
            )

    model.cuda()
    model.eval()
    print(
        f"Finished loading model weights. "
        f"missing={len(load_info['missing_keys'])}, "
        f"unexpected={len(load_info['unexpected_keys'])}"
    )
    return model

# 2. 实例化模型并加载权重
model = CLIPModel(use_spot_encoder=USE_SPOT_ENCODER).cuda()
model = load_trained_model(model_path, model)

# 3. 构建 DataLoader
test_loader = build_loaders_inference()

# 4. 一次性提取所有图像和基因特征
img_embeddings_all, spot_embeddings_all = extract_embeddings(model, test_loader)

# 5. 转成 numpy
img_embeddings_all = img_embeddings_all.numpy()
spot_embeddings_all = spot_embeddings_all.numpy()

print(f"总 Image 特征维度: {img_embeddings_all.shape}")
print(f"总 Spot 特征维度: {spot_embeddings_all.shape}")

# 6. 按照 datasize 切片并保存
if not os.path.exists(save_path):
    os.makedirs(save_path)

for i in range(4):
    index_start = sum(datasize[:i])
    index_end = sum(datasize[:i+1])

    image_embeddings = img_embeddings_all[index_start:index_end]
    spot_embeddings = spot_embeddings_all[index_start:index_end]

    print(f"切片 {i+1} - img: {image_embeddings.shape}, spot: {spot_embeddings.shape}")

    np.save(save_path + f"img_embeddings_{i+1}.npy", image_embeddings.T)
    np.save(save_path + f"spot_embeddings_{i+1}.npy", spot_embeddings.T)

print("🎉 所有切片特征均已成功保存！")

Loaded local weights from /root/disk2/runzhi/BLEEP/model.safetensors (missing=0, unexpected=2).


RuntimeError: Error(s) in loading state_dict for CLIPModel:
	size mismatch for image_projection.projection.weight: copying a param with shape torch.Size([256, 1024]) from checkpoint, the shape in current model is torch.Size([256, 2048]).

In [ ]:

#infer spot embeddings and expression
spot_expression1 = np.load("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/1/harmony_matrix.npy")
spot_expression2 = np.load("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/2/harmony_matrix.npy")
spot_expression3 = np.load("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/harmony_matrix.npy")
spot_expression4 = np.load("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/4/harmony_matrix.npy")

save_path = "/root/disk2/runzhi/BLEEP/result/embeddings/"
spot_embeddings1 = np.load(save_path + "spot_embeddings_1.npy")
spot_embeddings2 = np.load(save_path + "spot_embeddings_2.npy")
spot_embeddings3 = np.load(save_path + "spot_embeddings_3.npy")
spot_embeddings4 = np.load(save_path + "spot_embeddings_4.npy")
image_embeddings3 = np.load(save_path + "img_embeddings_3.npy")


#query
image_query = image_embeddings3
expression_gt = spot_expression3
spot_key = np.concatenate([spot_embeddings1, spot_embeddings2, spot_embeddings4], axis = 1)
expression_key = np.concatenate([spot_expression1, spot_expression2, spot_expression4], axis = 1)



In [ ]:
import pandas as pd

method = "average"

save_path = "/root/disk2/runzhi/BLEEP/result/embeddings/"
if image_query.shape[1] != 256:
    image_query = image_query.T
    print("image query shape: ", image_query.shape)
if expression_gt.shape[0] != image_query.shape[0]:
    expression_gt = expression_gt.T
    print("expression_gt shape: ", expression_gt.shape)
if spot_key.shape[1] != 256:
    spot_key = spot_key.T
    print("spot_key shape: ", spot_key.shape)
if expression_key.shape[0] != spot_key.shape[0]:
    expression_key = expression_key.T
    print("expression_key shape: ", expression_key.shape)

if method == "simple":
    indices = find_matches(spot_key, image_query, top_k=1)
    matched_spot_embeddings_pred = spot_key[indices[:,0],:]
    print("matched spot embeddings pred shape: ", matched_spot_embeddings_pred.shape)
    matched_spot_expression_pred = expression_key[indices[:,0],:]
    print("matched spot expression pred shape: ", matched_spot_expression_pred.shape)

if method == "average":
    print("finding matches, using average of top 50 expressions")
    indices = find_matches(spot_key, image_query, top_k=50)
    matched_spot_embeddings_pred = np.zeros((indices.shape[0], spot_key.shape[1]))
    matched_spot_expression_pred = np.zeros((indices.shape[0], expression_key.shape[1]))
    for i in range(indices.shape[0]):
        matched_spot_embeddings_pred[i,:] = np.average(spot_key[indices[i,:],:], axis=0)
        matched_spot_expression_pred[i,:] = np.average(expression_key[indices[i,:],:], axis=0)
    
    print("matched spot embeddings pred shape: ", matched_spot_embeddings_pred.shape)
    print("matched spot expression pred shape: ", matched_spot_expression_pred.shape)

if method == "weighted_average":
    print("finding matches, using weighted average of top 50 expressions")
    indices = find_matches(spot_key, image_query, top_k=50)
    matched_spot_embeddings_pred = np.zeros((indices.shape[0], spot_key.shape[1]))
    matched_spot_expression_pred = np.zeros((indices.shape[0], expression_key.shape[1]))
    for i in range(indices.shape[0]):
        a = np.sum((spot_key[indices[i,0],:] - image_query[i,:])**2) #the smallest MSE
        weights = np.exp(-(np.sum((spot_key[indices[i,:],:] - image_query[i,:])**2, axis=1)-a+1))
        if i == 0:
            print("weights: ", weights)
        matched_spot_embeddings_pred[i,:] = np.average(spot_key[indices[i,:],:], axis=0, weights=weights)
        matched_spot_expression_pred[i,:] = np.average(expression_key[indices[i,:],:], axis=0, weights=weights)
    
    print("matched spot embeddings pred shape: ", matched_spot_embeddings_pred.shape)
    print("matched spot expression pred shape: ", matched_spot_expression_pred.shape)



true = expression_gt
pred = matched_spot_expression_pred

print(pred.shape)
print(true.shape)

#genewise correlation
corr = np.zeros(pred.shape[0])
for i in range(pred.shape[0]):
    corr[i] = np.corrcoef(pred[i,:], true[i,:],)[0,1]
corr = corr[~np.isnan(corr)]
print("Mean correlation across cells: ", np.mean(corr))

corr = np.zeros(pred.shape[1])
for i in range(pred.shape[1]):
    corr[i] = np.corrcoef(pred[:,i], true[:,i],)[0,1]
corr = corr[~np.isnan(corr)]
print("number of non-zero genes: ", corr.shape[0])
print("max correlation: ", np.max(corr))
ind = np.argsort(np.sum(true, axis=0))[-50:]
print("mean correlation highly expressed genes: ", np.mean(corr[ind]))
ind = np.argsort(np.var(true, axis=0))[-50:]
print("mean correlation highly variable genes: ", np.mean(corr[ind]))

#marker genes
marker_gene_list = [ "HAL", "CYP3A4", "VWF", "SOX9", "KRT7", "ANXA4", "ACTA2", "DCN"] #markers from macparland paper
gene_names = pd.read_csv("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/features.tsv", header=None, sep="\t").iloc[:,1].values
hvg_b = np.load("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/hvg_union.npy")
marker_gene_ind = np.zeros(len(marker_gene_list))
for i in range(len(marker_gene_list)):
    marker_gene_ind[i] = np.where(gene_names[hvg_b] == marker_gene_list[i])[0]
print("mean correlation marker genes: ", np.mean(corr[marker_gene_ind.astype(int)]))

if save_path != "":
    np.save(save_path + "matched_spot_embeddings_pred.npy", matched_spot_embeddings_pred.T)
    np.save(save_path + "matched_spot_expression_pred.npy", matched_spot_expression_pred.T)

In [ ]:
#construct heatmap of the GGC matrix
expression_gt = np.load("/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/harmony_matrix.npy")
matched_spot_expression_pred_1 = np.load("/root/disk2/runzhi/BLEEP/result/embeddings/matched_spot_expression_pred.npy")

# matched_spot_expression_pred_2 = sc.read_h5ad('data/from_collab/harmony_C1_HisToGene_adata_pred.h5ad')
# matched_spot_expression_pred_2 = matched_spot_expression_pred_2.X.T
# matched_spot_expression_pred_3 = sc.read_h5ad('data/from_collab/harmony_C1_STNet_adata_pred.h5ad')
# matched_spot_expression_pred_3 = matched_spot_expression_pred_3.X.T

print(expression_gt.shape)
print(matched_spot_expression_pred_1.shape)


In [ ]:
#plot heatmap of top 50 genes ranked by mean
def plot_heatmap(expression_gt, matched_spot_expression_pred, top_k=50):
    #take mean of expression
    mean = np.mean(expression_gt, axis=1)
    #take ind of top 100
    ind = np.argpartition(mean, -top_k)[-top_k:]

    # Compute the correlation matrix
    corr_matrix = np.corrcoef(expression_gt[ind,:])
    dendrogram = hierarchy.dendrogram(hierarchy.linkage(corr_matrix, method='ward'), no_plot=True)
    cluster_idx = dendrogram['leaves']

    corr_matrix = np.corrcoef(matched_spot_expression_pred[ind,:])
    corr_matrix = corr_matrix[cluster_idx, :]
    corr_matrix = corr_matrix[:, cluster_idx]

    # Reorder the correlation matrix and plot the heatmap
    plt.figure(dpi=300, figsize=(5,5))
    sns.heatmap(corr_matrix, cmap='viridis', xticklabels=False, yticklabels=False, cbar= False, vmin=-1, vmax=1)

plot_heatmap(expression_gt, expression_gt, top_k=50)
plot_heatmap(expression_gt, matched_spot_expression_pred_1, top_k=50)
# plot_heatmap(expression_gt, matched_spot_expression_pred_2, top_k=50)
# plot_heatmap(expression_gt, matched_spot_expression_pred_3, top_k=50)




In [ ]:
import numpy as np

# 确保 true 和 pred 已经从前面的代码中继承 (true = expression_gt, pred = matched_spot_expression_pred)

print("="*40)
print("1. Overall Metrics (整体指标)")
print("="*40)
overall_mse = np.mean((true - pred) ** 2)
print(f"Overall MSE: {overall_mse:.4f}")

print("\n" + "="*40)
print("2. Spot-wise / Cell-wise Metrics (细胞/区域级别)")
print("="*40)
# 计算每个 Spot 的 MSE 并求平均
spot_mse = np.mean((true - pred) ** 2, axis=1)
print(f"Mean Spot-wise MSE: {np.mean(spot_mse):.4f}")

# 计算每个 Spot 的 PCC 并求平均
spot_pcc = np.zeros(pred.shape[0])
for i in range(pred.shape[0]):
    spot_pcc[i] = np.corrcoef(pred[i, :], true[i, :])[0, 1]
spot_pcc = spot_pcc[~np.isnan(spot_pcc)] # 过滤掉除零导致的 NaN
print(f"Mean Spot-wise PCC: {np.mean(spot_pcc):.4f}")

print("\n" + "="*40)
print("3. Gene-wise Metrics (基因级别)")
print("="*40)
# 计算每个基因的 MSE 并求平均
gene_mse = np.mean((true - pred) ** 2, axis=0)
print(f"Mean Gene-wise MSE: {np.mean(gene_mse):.4f}")

# 计算每个基因的 PCC 并求平均
gene_pcc = np.zeros(pred.shape[1])
for i in range(pred.shape[1]):
    gene_pcc[i] = np.corrcoef(pred[:, i], true[:, i])[0, 1]
gene_pcc = gene_pcc[~np.isnan(gene_pcc)] # 过滤掉除零导致的 NaN
print(f"Mean Gene-wise PCC: {np.mean(gene_pcc):.4f}")
print("="*40)

In [ ]:
# 计算方差并提取 Top 1000 的 HVG
import numpy as np
variances = np.var(true, axis=0)
top_1k_hvg_indices = np.argsort(variances)[-1000:]

# 提取这些有意义基因的 PCC 并过滤 NaN
hvg_1k_corr = corr[top_1k_hvg_indices]
hvg_1k_corr = hvg_1k_corr[~np.isnan(hvg_1k_corr)]

print(f"🔥 GSE240429 Top-1000 HVG Mean PCC: {np.mean(hvg_1k_corr):.4f} 🔥")